# FitSense Data Analysis

This notebook is the reproducible academic analysis for FitSense. It uses the supplied `dailyActivity_merged.csv` public Fitbit sample. The generated `FitnessLevel` target is a transparent activity-threshold label and is not a clinical outcome.

## 1. Objective, source, and configuration

The objective is to describe daily activity patterns, test an association between activity groups, compare leakage-safe supervised classifiers, and explore activity profiles with K-Means and PCA.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from analysis import load_and_clean_data, run_analysis

sns.set_theme(style='whitegrid')
frame, quality = load_and_clean_data()


## 2. Data acquisition and quality assessment

In [ ]:
print(quality)
display(frame.head())
display(frame.describe(include='all').transpose())

## 3. Cleaning, transformation, and descriptive statistics

The shared pipeline validates dates and numeric fields, removes duplicate participant-date rows, rejects invalid non-positive calorie records, and engineers active minutes and threshold labels.

In [ ]:
display(frame[['TotalSteps', 'Calories', 'ActivityScore', 'SedentaryMinutes']].describe().round(2))
print(frame['FitnessLevel'].value_counts())

## 4. Exploratory data analysis (six required views)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.histplot(frame, x='TotalSteps', bins=30, ax=axes[0, 0], color='#2dd4bf').set(title='Distribution of daily steps', xlabel='Steps per day')
sns.heatmap(frame[['TotalSteps', 'Calories', 'ActivityScore', 'SedentaryMinutes']].corr(), annot=True, cmap='vlag', ax=axes[0, 1]).set_title('Activity feature correlations')
sns.boxplot(data=frame[['TotalSteps', 'Calories', 'SedentaryMinutes']], ax=axes[0, 2]).set_title('Outlier review by IQR')
sns.scatterplot(data=frame, x='TotalSteps', y='Calories', hue='FitnessLevel', ax=axes[1, 0]).set(title='Steps and calories', xlabel='Steps per day', ylabel='Calories burned')
sns.countplot(data=frame, x='FitnessLevel', order=['Low', 'Medium', 'High'], ax=axes[1, 1], palette='viridis', hue='FitnessLevel', legend=False).set_title('Rule-based fitness labels')
sns.scatterplot(data=frame, x='TotalSteps', y='SedentaryMinutes', hue='FitnessLevel', ax=axes[1, 2]).set(title='Activity versus sedentary time', xlabel='Steps per day', ylabel='Sedentary minutes')
plt.tight_layout()
plt.show()

## 5. Statistical hypothesis testing

The Mann–Whitney U test compares step-count distributions between the Low and High threshold groups. It is non-parametric and appropriate when normality is not assumed.

In [ ]:
results = run_analysis(); print(results['statistical_test'])

## 6. Supervised learning, validation, and error analysis

Each estimator is inside a scikit-learn pipeline. Imputation and scaling are fitted within each cross-validation fold. A stratified five-fold split is used for these daily records; participant-grouped validation would be preferable for a deployment dataset with repeated users.

In [ ]:
pd.DataFrame(results['models']['comparison'])

## 7. K-Means, PCA, and cluster profiles

In [ ]:
display(pd.DataFrame(results['clustering']['scores']))
display(pd.DataFrame(results['clustering']['profiles']))
points = pd.DataFrame(results['clustering']['points'])
sns.scatterplot(data=points, x='x', y='y', hue='cluster', palette='viridis').set(title='PCA projection of K-Means profiles', xlabel='Principal component 1', ylabel='Principal component 2')
plt.show()

## 8. Conclusions, limitations, and recommendations

Use the printed results rather than copying static scores. The principal limitations are observational sampling, deterministic labels derived from activity thresholds, repeated participant records, and lack of clinical outcomes. Recommendations are wellness-oriented only; the application must not be used for diagnosis.